# Offline RAG Generation Prototype

## Decentralized Hiking AI


In [24]:
from pathlib import Path
import sqlite3
import pandas as pd
import re
import json

current = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:

    if (
        (candidate / "research").is_dir()
        and
        (candidate / "pubspec.yaml").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

RESEARCH_DIR = PROJECT_ROOT / "research"

RAG_DIR = (
    RESEARCH_DIR
    / "rag"
)

DATABASE_FILE = (
    RAG_DIR
    / "database"
    / "offline_hiking_knowledge.db"
)

EVALUATION_DIR = (
    RAG_DIR
    / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("OFFLINE RAG GENERATION ENVIRONMENT")
print("=" * 60)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Database exists:",
    DATABASE_FILE.exists()
)

print(
    "Database:",
    DATABASE_FILE
)

OFFLINE RAG GENERATION ENVIRONMENT
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
Database exists: True
Database: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db


In [25]:
connection = sqlite3.connect(
    DATABASE_FILE
)

cursor = connection.cursor()

database_tables = pd.read_sql_query(
    """
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY name
    """,
    connection
)

document_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM documents
    """
).fetchone()[0]

chunk_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM knowledge_chunks
    """
).fetchone()[0]

fts_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM knowledge_chunks_fts
    """
).fetchone()[0]

print("OFFLINE DATABASE CONNECTION")
print("=" * 60)

print(
    "Documents:",
    document_count
)

print(
    "Knowledge chunks:",
    chunk_count
)

print(
    "FTS5 chunks:",
    fts_count
)

print()

display(
    database_tables
)

OFFLINE DATABASE CONNECTION
Documents: 25
Knowledge chunks: 1779
FTS5 chunks: 1779



,name,type
0,documents,table
1,knowledge_chunks,table
2,knowledge_chunks_fts,table
3,knowledge_chunks_fts_config,table
4,knowledge_chunks_fts_content,table
5,knowledge_chunks_fts_data,table
6,knowledge_chunks_fts_docsize,table
7,knowledge_chunks_fts_idx,table


## Generation-Time Retrieval

In [26]:
def build_fts_query(question):

    tokens = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    stop_words = {
        "a", "an", "the",
        "and", "or",
        "if", "is", "are",
        "to", "of", "in",
        "on", "for",
        "what", "how",
        "should", "do",
        "i", "me", "my",
        "someone", "hiker"
    }

    useful_tokens = [
        token
        for token in tokens
        if (
            token not in stop_words
            and
            len(token) > 2
        )
    ]

    if not useful_tokens:
        return None

    return " OR ".join(
        f'"{token}"'
        for token in useful_tokens
    )


def retrieve_for_generation(
    question,
    category,
    limit=4
):

    fts_query = build_fts_query(
        question
    )

    if not fts_query:
        return pd.DataFrame()

    sql = """
    SELECT
        kc.chunk_id,
        kc.source_id,
        d.title,
        d.organization,
        d.source_url,
        kc.categories,
        kc.content,
        bm25(knowledge_chunks_fts) AS rank

    FROM knowledge_chunks_fts

    JOIN knowledge_chunks kc
        ON kc.chunk_id =
           knowledge_chunks_fts.chunk_id

    JOIN documents d
        ON d.source_id =
           kc.source_id

    WHERE
        knowledge_chunks_fts MATCH ?

        AND
        kc.categories LIKE ?

    ORDER BY rank

    LIMIT ?
    """

    return pd.read_sql_query(
        sql,
        connection,
        params=[
            fts_query,
            f"%{category}%",
            limit
        ]
    )

In [27]:
test_question = (
    "What should I do if a hiker "
    "is bitten by a snake?"
)

test_category = "snakebite"

generation_context_test = (
    retrieve_for_generation(
        question=test_question,
        category=test_category,
        limit=4
    )
)

print("GENERATION RETRIEVAL TEST")
print("=" * 60)

print(
    "Question:",
    test_question
)

print(
    "Category:",
    test_category
)

print(
    "Results:",
    len(
        generation_context_test
    )
)

display(
    generation_context_test[
        [
            "chunk_id",
            "source_id",
            "organization",
            "rank"
        ]
    ]
)

GENERATION RETRIEVAL TEST
Question: What should I do if a hiker is bitten by a snake?
Category: snakebite
Results: 4


,chunk_id,source_id,organization,rank
0,CHK_01692,EXP_002,World Health Organization (WHO),-11.545228
1,CHK_01685,EXP_001,Sri Lanka Medical Association (SLMA),-11.166481
2,CHK_01684,EXP_001,Sri Lanka Medical Association (SLMA),-10.949725
3,CHK_01693,EXP_002,World Health Organization (WHO),-8.388182


## Authoritative Context Construction

In [28]:
def build_grounded_context(
    retrieved_chunks,
    max_chars=7000
):

    if retrieved_chunks.empty:
        return ""

    context_parts = []

    used_chars = 0

    for _, row in (
        retrieved_chunks.iterrows()
    ):

        block = (
            f"[SOURCE {row['source_id']}]\n"
            f"Organization: "
            f"{row['organization']}\n"
            f"Title: {row['title']}\n"
            f"Evidence:\n"
            f"{row['content'].strip()}\n"
        )

        if (
            used_chars
            + len(block)
            > max_chars
        ):
            break

        context_parts.append(
            block
        )

        used_chars += len(block)

    return "\n---\n".join(
        context_parts
    )


grounded_context = (
    build_grounded_context(
        generation_context_test
    )
)

print("GROUNDED CONTEXT")
print("=" * 60)

print(
    "Characters:",
    len(
        grounded_context
    )
)

print()

print(
    grounded_context[:5000]
)

GROUNDED CONTEXT
Characters: 4835

[SOURCE EXP_002]
Organization: World Health Organization (WHO)
Title: Snakebite envenoming (WHO health topic)
Evidence:
lead to permanent disability and even limb amputation. Children are at higher risk of severe effects due to lower body mass than adults. Most deaths and serious consequences from snake bites are entirely preventable by making safe and effective antivenoms more widely available and accessible, particularly in high-risk areas. High quality snake antivenoms are the only effective treatment to prevent or reverse most of the venomous effects of snake bites. Treatment Snake antivenoms are effective treatments to prevent or reverse most of the harmful effects of snakebite envenoming and are included in the WHO list of essential medicines. The availability and accessibility of these antivenoms, along with raising awareness on primary prevention methods among communities and health workers, are the best ways to limit serious consequences and 

In [29]:
def build_rag_prompt(
    question,
    category,
    context
):

    return f"""
You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weather, rescue, or emergency advice.

2. If the supplied evidence is insufficient to answer the question,
   explicitly say that the offline knowledge base does not contain enough
   information.

3. Prefer instructions directly supported by the retrieved sources.

4. Do not treat foreign emergency telephone numbers as valid for Sri Lanka
   unless the supplied evidence explicitly identifies them as Sri Lankan.

5. Do not recommend unsafe traditional remedies or unsupported treatments.

6. For potentially serious medical emergencies, clearly communicate the need
   for professional medical assistance when supported by the evidence.

7. Keep the answer concise and practical for a hiker.

8. Do not claim to diagnose the person.

9. At the end include:
   Sources used: <source IDs>

CATEGORY:
{category}

QUESTION:
{question}

AUTHORITATIVE CONTEXT:
{context}

ANSWER:
""".strip()


snakebite_prompt = build_rag_prompt(
    question=test_question,
    category=test_category,
    context=grounded_context
)

print("OFFLINE RAG PROMPT")
print("=" * 60)

print(
    snakebite_prompt[:8000]
)

OFFLINE RAG PROMPT
You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weather, rescue, or emergency advice.

2. If the supplied evidence is insufficient to answer the question,
   explicitly say that the offline knowledge base does not contain enough
   information.

3. Prefer instructions directly supported by the retrieved sources.

4. Do not treat foreign emergency telephone numbers as valid for Sri Lanka
   unless the supplied evidence explicitly identifies them as Sri Lankan.

5. Do not recommend unsafe traditional remedies or unsupported treatments.

6. For potentially serious medical emergencies, clearly communicate the need
   for professional medical assistance when supported by the evidence.

7. Keep the answer concise and practical for a hiker.

8. Do not claim to diagnose the person.

9. At the end include:
   Sources used: <sou

In [30]:
prompt_checks = {
    "contains_question":
        test_question
        in snakebite_prompt,

    "contains_context":
        len(
            grounded_context
        ) > 0,

    "contains_source_id":
        "[SOURCE "
        in snakebite_prompt,

    "contains_grounding_rule":
        "ONLY the authoritative evidence"
        in snakebite_prompt,

    "contains_localization_rule":
        "foreign emergency telephone numbers"
        in snakebite_prompt,

    "within_prototype_size":
        len(
            snakebite_prompt
        ) < 12000
}

print("RAG PROMPT VALIDATION")
print("=" * 60)

for check, result in (
    prompt_checks.items()
):
    print(
        f"{check}: {result}"
    )

print()

print(
    "All prompt checks passed:",
    all(
        prompt_checks.values()
    )
)

print(
    "Prompt characters:",
    len(
        snakebite_prompt
    )
)

RAG PROMPT VALIDATION
contains_question: True
contains_context: True
contains_source_id: True
contains_grounding_rule: True
contains_localization_rule: True
within_prototype_size: True

All prompt checks passed: True
Prompt characters: 5940


## Offline Language Model

In [31]:
MODEL_DIR = (
    RAG_DIR
    / "models"
)

MODEL_FILE = (
    MODEL_DIR
    / "Qwen3.5-0.8B-Q4_0.gguf"
)

print("OFFLINE MODEL")
print("=" * 60)

print(
    "Model exists:",
    MODEL_FILE.exists()
)

if MODEL_FILE.exists():

    model_size_mb = (
        MODEL_FILE.stat().st_size
        / (1024 ** 2)
    )

    print(
        "Model size:",
        f"{model_size_mb:.2f} MB"
    )

print(
    "Model path:",
    MODEL_FILE
)

OFFLINE MODEL
Model exists: True
Model size: 536.95 MB
Model path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf


In [32]:
from pathlib import Path
import subprocess

LLAMA_CLI = Path(
    "/opt/homebrew/bin/llama-cli"
)

print("LLAMA.CPP RUNTIME")
print("=" * 60)

print(
    "llama-cli:",
    LLAMA_CLI
)

print(
    "Executable exists:",
    LLAMA_CLI.exists()
)

if not LLAMA_CLI.exists():
    raise RuntimeError(
        f"llama-cli was not found at {LLAMA_CLI}"
    )

version_test = subprocess.run(
    [
        str(LLAMA_CLI),
        "--version"
    ],
    capture_output=True,
    text=True
)

print()
print(
    version_test.stdout[:2000]
)

if version_test.stderr:
    print(
        version_test.stderr[:1000]
    )

LLAMA.CPP RUNTIME
llama-cli: /opt/homebrew/bin/llama-cli
Executable exists: True


version: 0.7.0 (build 11669, commit 620d2225c)
built with AppleClang 21.0.0.21000334 for Darwin arm64



## First Fully Local RAG Generation Test

In [33]:
import subprocess
import time

generation_command = [
    str(LLAMA_CLI),

    "-m",
    str(MODEL_FILE),

    "-c",
    "4096",

    "-n",
    "300",

    "--temp",
    "0.2",

    "-p",
    snakebite_prompt
]

print("FIRST OFFLINE RAG GENERATION")
print("=" * 60)

print(
    "Question:",
    test_question
)

print(
    "Category:",
    test_category
)

print(
    "Retrieved chunks:",
    len(generation_context_test)
)

print()

start_time = time.perf_counter()

generation_result = subprocess.run(
    generation_command,
    capture_output=True,
    text=True,
    timeout=300
)

generation_seconds = (
    time.perf_counter()
    - start_time
)

print(
    "Return code:",
    generation_result.returncode
)

print(
    "Generation time:",
    f"{generation_seconds:.2f} seconds"
)

print()

print("MODEL OUTPUT")
print("=" * 60)

print(
    generation_result.stdout[-6000:]
)

if generation_result.returncode != 0:

    print()
    print("STDERR")
    print("=" * 60)

    print(
        generation_result.stderr[-5000:]
    )

FIRST OFFLINE RAG GENERATION
Question: What should I do if a hiker is bitten by a snake?
Category: snakebite
Retrieved chunks: 4



KeyboardInterrupt: 

## Runtime Diagnostic

In [ ]:
import subprocess
import time

diagnostic_command = [
    str(LLAMA_CLI),

    "-m",
    str(MODEL_FILE),

    "-c",
    "1024",

    "-n",
    "20",

    "--temp",
    "0.2",

    "-p",
    "Reply with exactly: Offline model working."
]

print("MINIMAL LOCAL MODEL TEST")
print("=" * 60)

start_time = time.perf_counter()

try:

    diagnostic_result = subprocess.run(
        diagnostic_command,
        capture_output=True,
        text=True,
        timeout=120
    )

    diagnostic_seconds = (
        time.perf_counter()
        - start_time
    )

    print(
        "Return code:",
        diagnostic_result.returncode
    )

    print(
        "Time:",
        f"{diagnostic_seconds:.2f} seconds"
    )

    print()

    print("STDOUT")
    print("-" * 60)

    print(
        diagnostic_result.stdout[-3000:]
    )

    print()

    print("STDERR")
    print("-" * 60)

    print(
        diagnostic_result.stderr[-3000:]
    )

except subprocess.TimeoutExpired as error:

    diagnostic_seconds = (
        time.perf_counter()
        - start_time
    )

    print(
        "DIAGNOSTIC TIMED OUT"
    )

    print(
        "Time:",
        f"{diagnostic_seconds:.2f} seconds"
    )

    print()

    print("Partial STDOUT:")
    print(
        error.stdout[-3000:]
        if error.stdout
        else "None"
    )

    print()

    print("Partial STDERR:")
    print(
        error.stderr[-3000:]
        if error.stderr
        else "None"
    )

MINIMAL LOCAL MODEL TEST
DIAGNOSTIC TIMED OUT
Time: 120.04 seconds

Partial STDOUT:
b'\n\nLoading model... |\x08-\x08\\\x08|\x08/\x08-\x08\\\x08|\x08/\x08-\x08\\\x08|\x08 \x08\n\n\xe2\x96\x84\xe2\x96\x84 \xe2\x96\x84\xe2\x96\x84\n\xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88\n\xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88  \xe2\x96\x80\xe2\x96\x80\xe2\x96\x88\xe2\x96\x84 \xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x84\xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x84  \xe2\x96\x80\xe2\x96\x80\xe2\x96\x88\xe2\x96\x84    \xe2\x96\x84\xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x84 \xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x88\xe2\x96\x84\n\xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88 \xe2\x96\x84\xe2\x96\x88\xe2\x96\x80\xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88 \xe2\x96\x88\xe2\x96\x88 \xe2\x96\x84\xe2\x96\x88\xe2\x96\x80\xe2\x96\x88\xe2\x96\x88    \xe2\x96\x88\xe2\x96\x88    \xe2\x9

## Non-Interactive Generation Configuration

In [ ]:
import subprocess
import time

single_prompt_command = [
    str(LLAMA_CLI),

    "-m",
    str(MODEL_FILE),

    "-c",
    "1024",

    "-n",
    "40",

    "--temp",
    "0.2",

    "-st",

    "-p",
    "Reply with exactly: Offline model working."
]

print("SINGLE-TURN MODEL TEST")
print("=" * 60)

start_time = time.perf_counter()

try:

    single_prompt_result = subprocess.run(
        single_prompt_command,
        capture_output=True,
        text=True,
        timeout=120
    )

    single_prompt_seconds = (
        time.perf_counter()
        - start_time
    )

    print(
        "Return code:",
        single_prompt_result.returncode
    )

    print(
        "Time:",
        f"{single_prompt_seconds:.2f} seconds"
    )

    print()

    print("STDOUT")
    print("-" * 60)

    print(
        single_prompt_result.stdout[-3000:]
    )

    print()

    print("STDERR")
    print("-" * 60)

    print(
        single_prompt_result.stderr[-2000:]
    )

except subprocess.TimeoutExpired as error:

    print(
        "SINGLE-TURN TEST TIMED OUT"
    )

    print(
        "Partial STDOUT:",
        error.stdout
    )

    print(
        "Partial STDERR:",
        error.stderr
    )

SINGLE-TURN MODEL TEST
Return code: 0
Time: 1.69 seconds

STDOUT
------------------------------------------------------------


Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Reply with exactly: Offline model working.

[Start thinking]

Thinking Process:

1.  **Analyze the Request:**
    *   Input: "Offline model working."
    *   Task: Reply with exactly

## First Grounded Offline RAG Answer

In [ ]:
def generate_offline_answer(
    prompt,
    max_tokens=300,
    context_size=4096,
    temperature=0.2,
    timeout_seconds=120
):

    command = [
        str(LLAMA_CLI),

        "-m",
        str(MODEL_FILE),

        "-c",
        str(context_size),

        "-n",
        str(max_tokens),

        "--temp",
        str(temperature),

        "-st",

        "-p",
        prompt
    ]

    start_time = time.perf_counter()

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success":
                result.returncode == 0,

            "return_code":
                result.returncode,

            "time_seconds":
                elapsed,

            "stdout":
                result.stdout,

            "stderr":
                result.stderr
        }

    except subprocess.TimeoutExpired as error:

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": False,
            "return_code": None,
            "time_seconds": elapsed,
            "stdout":
                error.stdout or "",

            "stderr":
                error.stderr or "",

            "timeout": True
        }


print(
    "Offline generation function ready."
)

Offline generation function ready.


In [ ]:
snakebite_generation = (
    generate_offline_answer(
        prompt=snakebite_prompt,
        max_tokens=300,
        context_size=4096,
        temperature=0.2,
        timeout_seconds=120
    )
)

print("SNAKEBITE OFFLINE RAG TEST")
print("=" * 60)

print(
    "Success:",
    snakebite_generation[
        "success"
    ]
)

print(
    "Return code:",
    snakebite_generation[
        "return_code"
    ]
)

print(
    "Generation time:",
    f"{snakebite_generation['time_seconds']:.2f} seconds"
)

print()

print("RAW MODEL OUTPUT")
print("=" * 60)

print(
    snakebite_generation[
        "stdout"
    ][-8000:]
)

if snakebite_generation[
    "stderr"
]:

    print()
    print("STDERR")
    print("=" * 60)

    print(
        snakebite_generation[
            "stderr"
        ][-3000:]
    )

SNAKEBITE OFFLINE RAG TEST
Success: True
Return code: 0
Generation time: 7.52 seconds

RAW MODEL OUTPUT


Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weath

In [ ]:
raw_output = (
    snakebite_generation[
        "stdout"
    ]
)

print("RAW OUTPUT INSPECTION")
print("=" * 60)

print(
    "Total characters:",
    len(raw_output)
)

print()

print("LAST 12000 CHARACTERS")
print("=" * 60)

print(
    raw_output[-12000:]
)

RAW OUTPUT INSPECTION
Total characters: 2709

LAST 12000 CHARACTERS


Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weather, rescue, or emergency advice.

2.

In [ ]:
RESULTS_DIR = (
    RAG_DIR
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

snakebite_raw_file = (
    RESULTS_DIR
    / "snakebite_first_offline_generation.txt"
)

snakebite_raw_file.write_text(
    raw_output,
    encoding="utf-8"
)

print("GENERATION OUTPUT SAVED")
print("=" * 60)

print(
    "File:",
    snakebite_raw_file
)

print(
    "Exists:",
    snakebite_raw_file.exists()
)

print(
    "Characters:",
    len(raw_output)
)

GENERATION OUTPUT SAVED
File: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/results/snakebite_first_offline_generation.txt
Exists: True
Characters: 2709


In [ ]:
output_lower = (
    raw_output.lower()
)

audit_terms = [
    "exp_001",
    "exp_002",
    "hospital",
    "immobil",
    "carry",
    "walk",
    "traditional",
    "1990",
    "911",
    "999",
    "112"
]

print("FIRST GENERATION CONTENT AUDIT")
print("=" * 60)

for term in audit_terms:

    print(
        f"{term}:",
        term in output_lower
    )

FIRST GENERATION CONTENT AUDIT
exp_001: True
exp_002: True
hospital: True
immobil: True
carry: False
walk: False
traditional: True
1990: False
911: False
999: False
112: False


## Subprocess-Safe Output Capture

In [ ]:
def generate_offline_answer(
    prompt,
    max_tokens=300,
    context_size=4096,
    temperature=0.2,
    timeout_seconds=120
):

    command = [
        str(LLAMA_CLI),

        "-m",
        str(MODEL_FILE),

        "-c",
        str(context_size),

        "-n",
        str(max_tokens),

        "--temp",
        str(temperature),

        "-st",

        "--simple-io",

        "--no-display-prompt",

        "--log-disable",

        "-p",
        prompt
    ]

    start_time = time.perf_counter()

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success":
                result.returncode == 0,

            "return_code":
                result.returncode,

            "time_seconds":
                elapsed,

            "stdout":
                result.stdout.strip(),

            "stderr":
                result.stderr.strip()
        }

    except subprocess.TimeoutExpired as error:

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": False,
            "return_code": None,
            "time_seconds": elapsed,
            "stdout":
                error.stdout or "",

            "stderr":
                error.stderr or "",

            "timeout": True
        }


print(
    "Subprocess-safe offline generator ready."
)

Subprocess-safe offline generator ready.


In [ ]:
clean_test = generate_offline_answer(
    prompt=(
        "Reply only with the words: "
        "Offline model working."
    ),
    max_tokens=30,
    context_size=1024,
    temperature=0.0,
    timeout_seconds=60
)

print("CLEAN OUTPUT TEST")
print("=" * 60)

print(
    "Success:",
    clean_test["success"]
)

print(
    "Return code:",
    clean_test["return_code"]
)

print(
    "Time:",
    f"{clean_test['time_seconds']:.2f} seconds"
)

print()

print("CAPTURED OUTPUT")
print("=" * 60)

print(
    repr(
        clean_test["stdout"]
    )
)

print()

print(
    "Output characters:",
    len(
        clean_test["stdout"]
    )
)

print(
    "STDERR characters:",
    len(
        clean_test["stderr"]
    )
)

CLEAN OUTPUT TEST
Success: True
Return code: 0
Time: 1.75 seconds

CAPTURED OUTPUT
'Loading model... \n\n▄▄ ▄▄\n██ ██\n██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄\n██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██\n██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀\n                                    ██    ██\n                                    ▀▀    ▀▀\n\nbuild      : b11669-620d2225c\nmodel      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf\nftype      : Q4_0\nmodalities : text\n\navailable commands:\n  /exit or Ctrl+C     stop or exit\n  /regen              regenerate the last response\n  /clear              clear the chat history\n  /read <file>        add a text file\n  /glob <pattern>     add text files using globbing pattern\n\n\n\n> Reply only with the words: Offline model working.\n\n[Start thinking]\n\nThinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: "Offline model working."\n    *   Constraint:\n\n[ Prompt: 302.

In [ ]:
snakebite_generation_clean = (
    generate_offline_answer(
        prompt=snakebite_prompt,
        max_tokens=300,
        context_size=4096,
        temperature=0.2,
        timeout_seconds=120
    )
)

print("CLEAN SNAKEBITE RAG GENERATION")
print("=" * 60)

print(
    "Success:",
    snakebite_generation_clean[
        "success"
    ]
)

print(
    "Return code:",
    snakebite_generation_clean[
        "return_code"
    ]
)

print(
    "Generation time:",
    f"{snakebite_generation_clean['time_seconds']:.2f} seconds"
)

print()

print("GENERATED ANSWER")
print("=" * 60)

print(
    snakebite_generation_clean[
        "stdout"
    ]
)

CLEAN SNAKEBITE RAG GENERATION
Success: True
Return code: 0
Generation time: 8.05 seconds

GENERATED ANSWER
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, wea

In [ ]:
clean_answer = (
    snakebite_generation_clean[
        "stdout"
    ]
)

clean_answer_lower = (
    clean_answer.lower()
)

answer_audit = {
    "mentions_EXP_001":
        "exp_001"
        in clean_answer_lower,

    "mentions_EXP_002":
        "exp_002"
        in clean_answer_lower,

    "mentions_hospital":
        "hospital"
        in clean_answer_lower,

    "mentions_immobilization":
        "immobil"
        in clean_answer_lower,

    "mentions_carry":
        "carry"
        in clean_answer_lower,

    "mentions_traditional_methods":
        "traditional"
        in clean_answer_lower,

    "mentions_1990":
        "1990"
        in clean_answer_lower,

    "mentions_911":
        "911"
        in clean_answer_lower,

    "mentions_999":
        "999"
        in clean_answer_lower,

    "mentions_112":
        "112"
        in clean_answer_lower
}


print("GENERATED ANSWER AUDIT")
print("=" * 60)

for check, result in (
    answer_audit.items()
):

    print(
        f"{check}: {result}"
    )

GENERATED ANSWER AUDIT
mentions_EXP_001: True
mentions_EXP_002: True
mentions_hospital: True
mentions_immobilization: False
mentions_carry: False
mentions_traditional_methods: True
mentions_1990: False
mentions_911: False
mentions_999: False
mentions_112: False


## Disabling Reasoning for Safety Response Generation

In [34]:
no_think_test = generate_offline_answer(
    prompt=(
        "/no_think\n"
        "Reply only with exactly these words: "
        "Offline model working."
    ),
    max_tokens=60,
    context_size=1024,
    temperature=0.0,
    timeout_seconds=60
)

print("NO-THINK OUTPUT TEST")
print("=" * 60)

print(
    "Success:",
    no_think_test["success"]
)

print(
    "Return code:",
    no_think_test["return_code"]
)

print(
    "Time:",
    f"{no_think_test['time_seconds']:.2f} seconds"
)

print()

print("CAPTURED OUTPUT")
print("=" * 60)

print(
    no_think_test["stdout"]
)

NO-THINK OUTPUT TEST
Success: True
Return code: 0
Time: 2.77 seconds

CAPTURED OUTPUT
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> /no_think
Reply only with exactly these words: Offline model working.

[Start thinking]

Thinking Process:

1.  **Analyze the Request:**
    *   Input: "no_think" (This is a common prompt injection attempt to bypass safety f

## Persistent Local Inference Server

In [35]:
import json
import urllib.request
import urllib.error
import time

LLAMA_SERVER_URL = (
    "http://127.0.0.1:8080"
)

health_url = (
    LLAMA_SERVER_URL
    + "/health"
)

print("LOCAL LLM SERVER")
print("=" * 60)

try:

    with urllib.request.urlopen(
        health_url,
        timeout=10
    ) as response:

        health_body = (
            response
            .read()
            .decode("utf-8")
        )

        print(
            "HTTP status:",
            response.status
        )

        print(
            "Response:",
            health_body
        )

except Exception as error:

    print(
        "Server check failed:",
        repr(error)
    )

LOCAL LLM SERVER
HTTP status: 200
Response: {"status":"ok"}


In [36]:
def generate_with_local_server(
    prompt,
    max_tokens=220,
    temperature=0.1,
    timeout_seconds=120
):

    endpoint = (
        LLAMA_SERVER_URL
        + "/completion"
    )

    payload = {
        "prompt": prompt,
        "n_predict": max_tokens,
        "temperature": temperature,
        "stream": False
    }

    request = urllib.request.Request(
        endpoint,
        data=json.dumps(
            payload
        ).encode("utf-8"),
        headers={
            "Content-Type":
                "application/json"
        },
        method="POST"
    )

    start_time = (
        time.perf_counter()
    )

    try:

        with urllib.request.urlopen(
            request,
            timeout=timeout_seconds
        ) as response:

            response_body = (
                response
                .read()
                .decode("utf-8")
            )

            data = json.loads(
                response_body
            )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": True,
            "time_seconds": elapsed,
            "content":
                data.get(
                    "content",
                    ""
                ).strip(),
            "response":
                data
        }

    except Exception as error:

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": False,
            "time_seconds": elapsed,
            "content": "",
            "error": repr(error)
        }


print(
    "Local server generation function ready."
)

Local server generation function ready.


In [37]:
server_test = (
    generate_with_local_server(
        prompt=(
            "Answer directly without reasoning. "
            "Reply exactly with: "
            "Offline model working."
        ),
        max_tokens=60,
        temperature=0.0,
        timeout_seconds=60
    )
)

print("LOCAL SERVER GENERATION TEST")
print("=" * 60)

print(
    "Success:",
    server_test["success"]
)

print(
    "Time:",
    f"{server_test['time_seconds']:.2f} seconds"
)

print()

print("CONTENT")
print("=" * 60)

print(
    repr(
        server_test["content"]
    )
)

if not server_test["success"]:

    print()

    print(
        "Error:",
        server_test.get(
            "error"
        )
    )

LOCAL SERVER GENERATION TEST
Success: True
Time: 1.09 seconds

CONTENT
'Offline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.'


## End-to-End Offline RAG Generation

In [38]:
def build_grounded_rag_prompt(
    question,
    category,
    context
):

    return f"""
You are an offline hiking safety assistant.

Answer the QUESTION using ONLY the AUTHORITATIVE CONTEXT provided below.

STRICT RULES:
1. Do not use facts that are not present in the context.
2. Do not invent emergency telephone numbers.
3. Do not diagnose the hiker.
4. Give short, practical safety instructions.
5. If the context is insufficient, reply exactly:
   "The offline knowledge base does not contain enough information to answer this safely."
6. Do not show reasoning or a thinking process.
7. End the answer with:
   Sources: <source IDs actually used>
8. After the Sources line, output exactly:
   END_OF_ANSWER

CATEGORY:
{category}

QUESTION:
{question}

AUTHORITATIVE CONTEXT:
{context}

ANSWER:
""".strip()


snakebite_server_prompt = (
    build_grounded_rag_prompt(
        question=test_question,
        category=test_category,
        context=grounded_context
    )
)

print("SNAKEBITE SERVER PROMPT")
print("=" * 60)

print(
    "Characters:",
    len(snakebite_server_prompt)
)

print(
    "Contains EXP_001:",
    "EXP_001"
    in snakebite_server_prompt
)

print(
    "Contains EXP_002:",
    "EXP_002"
    in snakebite_server_prompt
)

SNAKEBITE SERVER PROMPT
Characters: 5574
Contains EXP_001: True
Contains EXP_002: True


In [39]:
snakebite_server_result = (
    generate_with_local_server(
        prompt=snakebite_server_prompt,
        max_tokens=300,
        temperature=0.1,
        timeout_seconds=60
    )
)

print("END-TO-END SNAKEBITE RAG")
print("=" * 60)

print(
    "Success:",
    snakebite_server_result[
        "success"
    ]
)

print(
    "Generation time:",
    f"{snakebite_server_result['time_seconds']:.2f} seconds"
)

print()

print("RAW GENERATED CONTENT")
print("=" * 60)

print(
    snakebite_server_result[
        "content"
    ]
)

END-TO-END SNAKEBITE RAG
Success: True
Generation time: 6.17 seconds

RAW GENERATED CONTENT
<think>

</think>

Sources: <source IDs actually used>
END_OF_ANSWER

What should I do if a hiker is bitten by a snake?

Immediately move away from the area where the bite occurred. Remove anything tight from around the bitten part of the body to avoid harm if swelling occurs. Reassure the victim, as most venomous snake bites do not cause immediate death. Immobilize the person completely and transport the person to a health facility as soon as possible. Apply pressure at the bite site with a pressure pad may be suitable in some cases. Avoid traditional first aid methods or herbal medicines. Paracetamol may be given for local pain (which can be severe). Vomiting may occur, so place the person on their left side in the recovery position. Closely monitor airway and breathing and be ready to resuscitate if necessary.

Sources: <source IDs actually used>
END_OF_ANSWER

Wait, I need to check the sourc

In [40]:
raw_snakebite_answer = (
    snakebite_server_result[
        "content"
    ]
)

if "END_OF_ANSWER" in raw_snakebite_answer:

    snakebite_answer = (
        raw_snakebite_answer
        .split(
            "END_OF_ANSWER",
            1
        )[0]
        .strip()
    )

else:

    snakebite_answer = (
        raw_snakebite_answer.strip()
    )


print("CLEAN SNAKEBITE ANSWER")
print("=" * 60)

print(
    snakebite_answer
)

CLEAN SNAKEBITE ANSWER
<think>

</think>

Sources: <source IDs actually used>


In [41]:
answer_lower = (
    snakebite_answer.lower()
)

snakebite_audit = {
    "answer_generated":
        len(
            snakebite_answer
        ) > 0,

    "mentions_EXP_001":
        "exp_001"
        in answer_lower,

    "mentions_EXP_002":
        "exp_002"
        in answer_lower,

    "mentions_hospital":
        "hospital"
        in answer_lower,

    "mentions_keep_still_or_immobilize":
        (
            "keep"
            in answer_lower
            and
            "still"
            in answer_lower
        )
        or
        "immobil"
        in answer_lower,

    "mentions_remove_tight_items":
        (
            "tight"
            in answer_lower
            or
            "ring"
            in answer_lower
            or
            "constrict"
            in answer_lower
        ),

    "mentions_avoid_traditional_methods":
        (
            "traditional"
            in answer_lower
            or
            "herbal"
            in answer_lower
        ),

    "foreign_911_present":
        "911"
        in answer_lower,

    "foreign_999_present":
        "999"
        in answer_lower,

    "foreign_112_present":
        "112"
        in answer_lower,

    "unsupported_1990_present":
        "1990"
        in answer_lower
}


print("SNAKEBITE ANSWER AUDIT")
print("=" * 60)

for check, result in (
    snakebite_audit.items()
):

    print(
        f"{check}: {result}"
    )

SNAKEBITE ANSWER AUDIT
answer_generated: True
mentions_EXP_001: False
mentions_EXP_002: False
mentions_hospital: False
mentions_keep_still_or_immobilize: False
mentions_remove_tight_items: False
mentions_avoid_traditional_methods: False
foreign_911_present: False
foreign_999_present: False
foreign_112_present: False
unsupported_1990_present: False


## Deterministic Source Attribution and Answer Extraction

In [43]:
snakebite_question = (
    "What should I do if a hiker is bitten by a snake?"
)

snakebite_category = "snakebite"

snakebite_results = retrieve_knowledge(
    question=snakebite_question,
    category=snakebite_category,
    limit=4
)

print("SNAKEBITE RETRIEVAL REFRESH")
print("=" * 60)

print(
    "Results:",
    len(snakebite_results)
)

display(
    snakebite_results[
        [
            "chunk_id",
            "source_id",
            "organization",
            "rank"
        ]
    ]
)

NameError: name 'retrieve_knowledge' is not defined

# Unified Offline RAG Pipeline

In [44]:
import sqlite3
import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT.name
    != "Decentralized-Hiking-AI"
    and
    PROJECT_ROOT.parent
    != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent


RAG_DIR = (
    PROJECT_ROOT
    / "research"
    / "rag"
)

DATABASE_FILE = (
    RAG_DIR
    / "database"
    / "offline_hiking_knowledge.db"
)


rag_connection = sqlite3.connect(
    DATABASE_FILE
)


print("UNIFIED RAG DATABASE")
print("=" * 60)

print(
    "Database:",
    DATABASE_FILE
)

print(
    "Exists:",
    DATABASE_FILE.exists()
)

print(
    "Connection ready:",
    rag_connection is not None
)

UNIFIED RAG DATABASE
Database: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db
Exists: True
Connection ready: True


In [45]:
def retrieve_offline_knowledge(
    question,
    category=None,
    limit=4
):

    search_terms = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    search_terms = [
        term
        for term in search_terms
        if len(term) >= 3
    ]

    fts_query = " OR ".join(
        search_terms
    )

    if not fts_query:
        return pd.DataFrame()

    if category:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.category,
            kc.content,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts
            MATCH ?

            AND kc.category = ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            category,
            limit
        )

    else:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.category,
            kc.content,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts
            MATCH ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            limit
        )

    return pd.read_sql_query(
        sql,
        rag_connection,
        params=params
    )


print(
    "Unified retrieval function ready."
)

Unified retrieval function ready.


In [46]:
def prepare_rag_context(
    results
):

    if results.empty:

        return "", []

    context_parts = []

    source_ids = list(
        dict.fromkeys(
            results[
                "source_id"
            ].tolist()
        )
    )

    for _, row in results.iterrows():

        context_parts.append(
            f"""
[SOURCE {row['source_id']}]
Organization: {row['organization']}
Title: {row['title']}
Evidence:
{row['content']}
""".strip()
        )

    context = (
        "\n\n---\n\n"
        .join(
            context_parts
        )
    )

    return (
        context,
        source_ids
    )


print(
    "Context builder ready."
)

Context builder ready.


In [48]:
offline_rag_answer(
    question,
    category
)

NameError: name 'offline_rag_answer' is not defined

In [47]:
offline_rag_answer(
    "What should I do if I am lost on a hiking trail?",
    "lost_navigation"
)

NameError: name 'offline_rag_answer' is not defined

In [49]:
offline_rag_answer(
    "How can I get emergency medical help in Sri Lanka?",
    "sri_lanka_emergency"
)

NameError: name 'offline_rag_answer' is not defined

## Unified Offline RAG Answer Function

In [50]:
def offline_rag_answer(
    question,
    category,
    limit=4
):

    # ---------------------------------
    # 1. Retrieve evidence from SQLite
    # ---------------------------------

    results = retrieve_offline_knowledge(
        question=question,
        category=category,
        limit=limit
    )

    if results.empty:

        return {
            "success": False,
            "question": question,
            "category": category,
            "answer": (
                "The offline knowledge base does not contain "
                "enough information to answer this safely."
            ),
            "source_ids": [],
            "retrieved_chunks": 0,
            "generation_time": 0
        }


    # ---------------------------------
    # 2. Build grounded context
    # ---------------------------------

    context, source_ids = (
        prepare_rag_context(
            results
        )
    )


    # ---------------------------------
    # 3. Build generation prompt
    # ---------------------------------

    prompt = f"""
You are an offline hiking safety assistant.

Answer the QUESTION using ONLY the AUTHORITATIVE CONTEXT.

Rules:
- Give concise practical guidance.
- Do not add information absent from the context.
- Do not invent emergency telephone numbers.
- Do not diagnose the hiker.
- Do not provide source IDs.
- Do not explain your reasoning.
- If the evidence is insufficient, output exactly:
  INSUFFICIENT_OFFLINE_EVIDENCE

QUESTION:
{question}

AUTHORITATIVE CONTEXT:
{context}

FINAL ANSWER:
""".strip()


    # ---------------------------------
    # 4. Local LLM generation
    # ---------------------------------

    generation = (
        generate_with_local_server(
            prompt=prompt,
            max_tokens=220,
            temperature=0.0,
            timeout_seconds=60
        )
    )


    if not generation["success"]:

        return {
            "success": False,
            "question": question,
            "category": category,
            "answer": (
                "Local generation failed."
            ),
            "source_ids": source_ids,
            "retrieved_chunks":
                len(results),
            "generation_time":
                generation[
                    "time_seconds"
                ]
        }


    # ---------------------------------
    # 5. Clean generated text
    # ---------------------------------

    answer = (
        generation["content"]
        .strip()
    )

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        answer,
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()


    # ---------------------------------
    # 6. Insufficient-evidence handling
    # ---------------------------------

    if (
        "INSUFFICIENT_OFFLINE_EVIDENCE"
        in answer
    ):

        answer = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )


    # ---------------------------------
    # 7. Deterministic provenance
    # ---------------------------------

    final_output = (
        answer
        + "\n\nSources: "
        + ", ".join(source_ids)
    )


    return {
        "success": True,
        "question": question,
        "category": category,
        "answer": answer,
        "final_output": final_output,
        "source_ids": source_ids,
        "retrieved_chunks":
            len(results),
        "generation_time":
            generation[
                "time_seconds"
            ],
        "retrieval_results":
            results
    }


print(
    "Unified offline RAG function ready."
)

Unified offline RAG function ready.


In [51]:
lost_test = (
    offline_rag_answer(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation"
    )
)

print("LOST / NAVIGATION RAG TEST")
print("=" * 60)

print(
    "Success:",
    lost_test["success"]
)

print(
    "Retrieved chunks:",
    lost_test[
        "retrieved_chunks"
    ]
)

print(
    "Source IDs:",
    lost_test[
        "source_ids"
    ]
)

print(
    "Generation time:",
    f"{lost_test['generation_time']:.2f} seconds"
)

print()

print("ANSWER")
print("=" * 60)

print(
    lost_test.get(
        "final_output",
        lost_test["answer"]
    )
)

DatabaseError: Execution failed on sql '
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.category,
            kc.content,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts
            MATCH ?

            AND kc.category = ?

        ORDER BY rank

        LIMIT ?
        ': no such column: kc.category

In [52]:
print("SQLITE DATABASE SCHEMA")
print("=" * 60)

tables_to_check = [
    "documents",
    "knowledge_chunks",
    "knowledge_chunks_fts"
]

for table in tables_to_check:

    print()
    print(f"TABLE: {table}")
    print("-" * 60)

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        rag_connection
    )

    print(
        schema[
            [
                "name",
                "type"
            ]
        ].to_string(
            index=False
        )
    )

SQLITE DATABASE SCHEMA

TABLE: documents
------------------------------------------------------------
           name type
      source_id TEXT
          title TEXT
   organization TEXT
     categories TEXT
     source_url TEXT
      file_type TEXT
     local_path TEXT
country_context TEXT

TABLE: knowledge_chunks
------------------------------------------------------------
              name    type
          chunk_id    TEXT
         source_id    TEXT
source_chunk_index INTEGER
        categories    TEXT
        word_count INTEGER
           content    TEXT

TABLE: knowledge_chunks_fts
------------------------------------------------------------
      name type
  chunk_id     
 source_id     
categories     
   content     


## Schema-Aligned Offline Retrieval

In [53]:
def retrieve_offline_knowledge(
    question,
    category=None,
    limit=4
):

    search_terms = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    # Remove very short terms that add little retrieval value.
    search_terms = [
        term
        for term in search_terms
        if len(term) >= 3
    ]

    # Remove duplicates while preserving order.
    search_terms = list(
        dict.fromkeys(
            search_terms
        )
    )

    if not search_terms:
        return pd.DataFrame()

    fts_query = " OR ".join(
        search_terms
    )

    if category:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.categories,
            kc.content,
            kc.word_count,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts MATCH ?

            AND (
                ',' || REPLACE(kc.categories, ' ', '') || ','
            ) LIKE ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            f"%,{category},%",
            limit
        )

    else:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.categories,
            kc.content,
            kc.word_count,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts MATCH ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            limit
        )

    return pd.read_sql_query(
        sql,
        rag_connection,
        params=params
    )


print(
    "Schema-aligned retrieval function ready."
)

Schema-aligned retrieval function ready.


In [54]:
lost_retrieval_test = (
    retrieve_offline_knowledge(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation",
        limit=4
    )
)

print("LOST / NAVIGATION RETRIEVAL")
print("=" * 60)

print(
    "Results:",
    len(lost_retrieval_test)
)

if not lost_retrieval_test.empty:

    display(
        lost_retrieval_test[
            [
                "chunk_id",
                "source_id",
                "organization",
                "categories",
                "rank"
            ]
        ]
    )

LOST / NAVIGATION RETRIEVAL
Results: 3


,chunk_id,source_id,organization,categories,rank
0,CHK_01740,EXP_011,Department of Conservation New Zealand (DOC),lost_navigation,-10.214064
1,CHK_01741,EXP_011,Department of Conservation New Zealand (DOC),lost_navigation,-8.643724
2,CHK_01739,EXP_011,Department of Conservation New Zealand (DOC),lost_navigation,-6.030610


In [55]:
lightning_retrieval_test = (
    retrieve_offline_knowledge(
        question=(
            "What should a hiker do "
            "during lightning?"
        ),
        category="lightning",
        limit=4
    )
)

print("LIGHTNING RETRIEVAL")
print("=" * 60)

print(
    "Results:",
    len(lightning_retrieval_test)
)

if not lightning_retrieval_test.empty:

    display(
        lightning_retrieval_test[
            [
                "chunk_id",
                "source_id",
                "organization",
                "categories",
                "rank"
            ]
        ]
    )

LIGHTNING RETRIEVAL
Results: 4


,chunk_id,source_id,organization,categories,rank
0,CHK_01767,EXP_014,UK Met Office,lightning,-7.990868
1,CHK_01766,EXP_014,UK Met Office,lightning,-7.966359
2,CHK_01770,EXP_014,UK Met Office,lightning,-3.124784
3,CHK_01774,EXP_014,UK Met Office,lightning,-2.484784


## Multi-Category End-to-End Offline RAG Validation

In [56]:
lost_rag_test = offline_rag_answer(
    question=(
        "What should I do if I am lost "
        "on a hiking trail?"
    ),
    category="lost_navigation"
)

print("END-TO-END LOST / NAVIGATION RAG")
print("=" * 60)

print(
    "Success:",
    lost_rag_test["success"]
)

print(
    "Retrieved chunks:",
    lost_rag_test["retrieved_chunks"]
)

print(
    "Source IDs:",
    lost_rag_test["source_ids"]
)

print(
    "Generation time:",
    f"{lost_rag_test['generation_time']:.2f} seconds"
)

print()
print("FINAL ANSWER")
print("=" * 60)

print(
    lost_rag_test.get(
        "final_output",
        lost_rag_test["answer"]
    )
)

END-TO-END LOST / NAVIGATION RAG
Success: True
Retrieved chunks: 3
Source IDs: ['EXP_011']
Generation time: 3.65 seconds

FINAL ANSWER
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_011


In [57]:
lightning_rag_test = offline_rag_answer(
    question=(
        "What should a hiker do "
        "during lightning?"
    ),
    category="lightning"
)

print("END-TO-END LIGHTNING RAG")
print("=" * 60)

print(
    "Success:",
    lightning_rag_test["success"]
)

print(
    "Retrieved chunks:",
    lightning_rag_test["retrieved_chunks"]
)

print(
    "Source IDs:",
    lightning_rag_test["source_ids"]
)

print(
    "Generation time:",
    f"{lightning_rag_test['generation_time']:.2f} seconds"
)

print()
print("FINAL ANSWER")
print("=" * 60)

print(
    lightning_rag_test.get(
        "final_output",
        lightning_rag_test["answer"]
    )
)

END-TO-END LIGHTNING RAG
Success: True
Retrieved chunks: 4
Source IDs: ['EXP_014']
Generation time: 3.99 seconds

FINAL ANSWER
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_014


In [58]:
multi_category_audit = pd.DataFrame(
    [
        {
            "test": "lost_navigation",
            "success":
                lost_rag_test["success"],
            "retrieved_chunks":
                lost_rag_test["retrieved_chunks"],
            "unique_sources":
                len(
                    lost_rag_test["source_ids"]
                ),
            "has_answer":
                bool(
                    lost_rag_test["answer"].strip()
                ),
            "has_sources":
                bool(
                    lost_rag_test["source_ids"]
                ),
            "generation_seconds":
                lost_rag_test["generation_time"]
        },

        {
            "test": "lightning",
            "success":
                lightning_rag_test["success"],
            "retrieved_chunks":
                lightning_rag_test["retrieved_chunks"],
            "unique_sources":
                len(
                    lightning_rag_test["source_ids"]
                ),
            "has_answer":
                bool(
                    lightning_rag_test["answer"].strip()
                ),
            "has_sources":
                bool(
                    lightning_rag_test["source_ids"]
                ),
            "generation_seconds":
                lightning_rag_test["generation_time"]
        }
    ]
)

print("MULTI-CATEGORY RAG AUDIT")
print("=" * 60)

display(
    multi_category_audit
)

print()

print(
    "All tests successful:",
    multi_category_audit[
        "success"
    ].all()
)

print(
    "All tests retrieved evidence:",
    (
        multi_category_audit[
            "retrieved_chunks"
        ] > 0
    ).all()
)

print(
    "All tests produced answers:",
    multi_category_audit[
        "has_answer"
    ].all()
)

print(
    "All tests have provenance:",
    multi_category_audit[
        "has_sources"
    ].all()
)

MULTI-CATEGORY RAG AUDIT


,test,success,retrieved_chunks,unique_sources,has_answer,has_sources,generation_seconds
0,lost_navigation,True,3,1,True,True,3.646305
1,lightning,True,4,1,True,True,3.987891



All tests successful: True
All tests retrieved evidence: True
All tests produced answers: True
All tests have provenance: True
